In [ ]:
# # install packages
# !pip install -qq matpower matpowercaseframes

In [ ]:
from importlib.metadata import PackageNotFoundError, version

import matpower

print(f"matpower version: {matpower.__version__}")

try:
    print(f"matlabengine version: {version('matlabengine')}")
except PackageNotFoundError as e:
    print(f"matlabengine not available: {e}")

matpower version: 8.1.0.2.2.4
matlabengine version: 25.2.2


In [ ]:
import pandas as pd
from matpowercaseframes.constants import COLUMNS

from matpower import start_instance

In [ ]:
try:
    m = start_instance(engine="matlab")
    MATLAB_AVAILABLE = True
except ImportError:
    MATLAB_AVAILABLE = False

## `runpf` wrapper to avoid `TypeError`

In [ ]:
DEFAULT_MPC_FIELDS = ["baseMVA", "version", "bus", "gen", "branch", "gencost"]


def run_matlab_cmd(cmd, m=None, fields=None, **kwargs):
    """
    Generic wrapper to run a MATPOWER command and extract struct fields using MATLAB
    backend.

    Parameters
    ----------
    cmd : str
        Full MATLAB command string, e.g. "runpf(mpc_, mpopt_)".
    m : matpower instance, optional
        If None, a new instance is started and shut down after.
    fields : list of str, optional
        Struct field names to extract from result. Defaults to DEFAULT_MPC_FIELDS.
    **kwargs :
        Variables injected into MATLAB workspace before running cmd.
        Keys become MATLAB variable names, e.g. mpc_=mpc, mpopt_=mpopt.
    """
    if fields is None:
        fields = DEFAULT_MPC_FIELDS

    if m is None:
        m = start_instance(engine="matlab")
        SHUTDOWN = True
    else:
        SHUTDOWN = False

    for var_name, value in kwargs.items():
        m.workspace[var_name] = value
    m.eval(f"r1_ = {cmd};", nargout=0)

    result = {}
    for field in fields:
        result[field] = m.eval(f"r1_.{field};", nargout=1)

    if SHUTDOWN:
        m.quit()

    return result

In [ ]:
if MATLAB_AVAILABLE:
    mpc = m.loadcase("case9")
    display(pd.DataFrame(mpc["gen"], columns=COLUMNS["gen"][: mpc["gen"].size[1]]))

,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
0,1.0,72.3,27.03,300.0,-300.0,1.040,100.0,1.0,250.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2.0,163.0,6.54,300.0,-300.0,1.025,100.0,1.0,300.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3.0,85.0,-10.95,300.0,-300.0,1.025,100.0,1.0,270.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
if MATLAB_AVAILABLE:
    mpc = run_matlab_cmd("runopf(mpc)", m=m, mpc=mpc)


MATPOWER Version 8.1.1-dev, 25-Sep-2025
Optimal Power Flow -- AC-polar-power formulation
MATPOWER Interior Point Solver -- MIPS, Version 1.5.2, 12-Jul-2025
 (using built-in linear solver)
Converged!
OPF successful

Converged in 0.25 seconds
Objective Function Value = 5296.69 $/hr
|     System Summary                                                           |

How many?                How much?              P (MW)            Q (MVAr)
---------------------    -------------------  -------------  -----------------
Buses              9     Total Gen Capacity     820.0        -900.0 to 900.0
Generators         3     On-line Capacity       820.0        -900.0 to 900.0
Committed Gens     3     Generation (actual)    318.3              -9.6
Loads              3     Load                   315.0             115.0
  Fixed            3       Fixed                315.0             115.0
  Dispatchable     0       Dispatchable          -0.0 of -0.0      -0.0
Shunts             0     Shunt (inj)    

In [ ]:
if MATLAB_AVAILABLE:
    display(pd.DataFrame(mpc["gen"], columns=COLUMNS["gen"]))

,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF,MU_PMAX,MU_PMIN,MU_QMAX,MU_QMIN
0,1.0,89.798614,12.938736,300.0,-300.0,1.099951,100.0,1.0,250.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2.0,134.320652,0.047730,300.0,-300.0,1.097363,100.0,1.0,300.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3.0,94.187439,-22.619730,300.0,-300.0,1.086627,100.0,1.0,270.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
